# Merge Hasil Extractive

Menggabungkan hasil tiga metode extractive (TF-IDF, TextRank, BERT) menjadi satu file `summary_gabungan_extractive.csv` untuk uji skenario. Merge by `global_id` agar tidak berantakan.

---
## 1. Load Ketiga Hasil Summary

In [ ]:
import pandas as pd

BASE = '/kaggle/input/datasets/NAMA_DATASET'

tfidf    = pd.read_csv(f'{BASE}/hasil_summary_tfidf.csv', encoding='utf-8-sig')
textrank = pd.read_csv(f'{BASE}/hasil_summary_textrank.csv', encoding='utf-8-sig')
bert     = pd.read_csv(f'{BASE}/hasil_summary_bert.csv', encoding='utf-8-sig')

print(f'TF-IDF   : {len(tfidf)} baris')
print(f'TextRank : {len(textrank)} baris')
print(f'BERT     : {len(bert)} baris')

---
## 2. Merge by global_id
Merge memakai `global_id` agar tiap ringkasan menempel pada artikel yang benar, meski urutan baris antar-file berbeda.

In [ ]:
# Merge by global_id agar tiap ringkasan selaras dengan artikel yang benar,
# meski urutan baris antar-file berbeda.
COMMON = ['global_id', 'title', 'category', 'lead_paragraph', 'body_word_count']

merged = tfidf[COMMON + ['summary_tf_idf']].copy()
merged = merged.merge(textrank[['global_id', 'summary_textrank']], on='global_id', how='inner')
merged = merged.merge(bert[['global_id', 'summary_bert']], on='global_id', how='inner')

merged = merged[['global_id', 'title', 'category',
                 'summary_tf_idf', 'summary_textrank', 'summary_bert',
                 'lead_paragraph', 'body_word_count']]

print(f'Hasil merge: {len(merged)} baris, {len(merged.columns)} kolom')
print(f'Kolom: {list(merged.columns)}')

---
## 3. Validasi Hasil Merge

In [ ]:
# Validasi: pastikan tidak ada data hilang atau kosong
print('Cek keselarasan & kualitas data:')
print(f'  Baris hasil merge      : {len(merged)}')
print(f'  global_id duplikat     : {merged["global_id"].duplicated().sum()}')
print(f'  Nilai kosong/NaN total : {merged.isna().sum().sum()}')
print()
print('Nilai kosong per kolom summary:')
for col in ['summary_tf_idf', 'summary_textrank', 'summary_bert']:
    n = merged[col].isna().sum() + (merged[col].astype(str).str.strip() == '').sum()
    print(f'  {col:<20}: {n}')

---
## 4. Simpan Hasil Gabungan

In [ ]:
merged.to_csv('summary_gabungan_extractive.csv', index=False, encoding='utf-8-sig')
print(f'Tersimpan: summary_gabungan_extractive.csv ({len(merged)} baris)')
print(f'\nContoh baris pertama:')
print(merged.iloc[0].to_string())